# Clase 9: Estadística Descriptiva — Distribuciones, Dispersión y Outliers

**Diplomado en Data Science Aplicada con Python para la Toma de Decisiones**  
Arca Continental Ecuador · UDLA · 2026

---

**Instructor:** Carlos Enrique Mosquera Trujillo  
**Contacto:** cmosquerat@unal.edu.co

---

### Contenido de esta sesión

| # | Tema | Duración |
|---|------|----------|
| 1 | ¿Qué es una distribución? Formas y simetría | 15 min |
| 2 | Tendencia central en profundidad: media, mediana, moda | 20 min |
| 3 | Medidas de dispersión: varianza, std, IQR, CV | 20 min |
| 4 | Outliers: detectarlos y decidir qué hacer | 15 min |
| 5 | La distribución Normal y la regla 68-95-99.7 | 15 min |
| 6 | De estadísticas a información: convertir números en decisiones | 10 min |
| 7 | Práctica integradora | 15 min |

---
## Recapitulación: Clase 8

En la clase anterior aprendimos a:
- Cargar y explorar un dataset con pandas
- Distinguir tipos de dato: numérico vs categórico
- Detectar y tratar NaN (media, mediana, moda)
- Crear histogramas y boxplots con seaborn

**Hoy profundizamos:** Vamos más allá de *describir* los datos. Aprenderemos a *interpretar* las estadísticas, entender cuándo cada medida es apropiada, y —lo más importante— cómo convertir números en **información para tomar decisiones**.

> **Pregunta guía de hoy:** Un número por sí solo no cuenta la historia completa. ¿Cómo sabemos *cuál* número usar y *qué* nos está diciendo realmente?

---
## 0. Preparación: Cargar el Dataset y Limpiar

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

# Mismo dataset de la Clase 8
url = ("https://raw.githubusercontent.com/cmosquerat/"
       "arca-diplomado/main/clase-08/supermarket_sales.csv")
df = pd.read_csv(url)

# Limpieza rápida (lo que aprendimos en Clase 8)
df["Date"] = pd.to_datetime(df["Date"])
df["Total"] = df["Total"].fillna(df["Total"].median())
df["Rating"] = df["Rating"].fillna(df["Rating"].mean())
df["Payment"] = df["Payment"].fillna(df["Payment"].mode()[0])

print(f"Dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"NaN restantes: {df.isna().sum().sum()}")
df.head(3)

---
## 1. ¿Qué Es una Distribución?

### El concepto más importante de la estadística

Un dataset no se resume con un solo número. La **distribución** es *cómo se reparten los valores* de una variable.

Imagina que mides la estatura de 1000 personas:
- ¿Cuántas miden 1.50m? ¿1.70m? ¿2.00m?
- ¿Se acumulan en el centro o están dispersas?
- ¿Hay un pico o varios?

La distribución responde **todas** estas preguntas — y determina **qué herramientas estadísticas son válidas**.

### Tres formas de verla

| Herramienta | Qué hace | Código |
|------------|----------|--------|
| **Histograma** | Divide en "bins" y cuenta cuántos valores caen en cada uno | `sns.histplot()` |
| **KDE** (Kernel Density Estimation) | Curva suave que estima la forma | `sns.kdeplot()` |
| **Ambos juntos** | La mejor vista inicial | `sns.histplot(kde=True)` |

> **KDE en una oración:** Es una versión suave del histograma. Donde hay más datos, la curva sube. Donde hay pocos, baja. No depende de cuántos "bins" elijas.

In [ ]:
# Tres formas de visualizar la distribución de Rating
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Solo histograma
sns.histplot(df["Rating"], bins=20, color="#2563EB", ax=axes[0])
axes[0].set_title("Solo Histograma", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Rating")

# 2. Solo KDE
sns.kdeplot(df["Rating"], fill=True, color="#7C3AED", ax=axes[1])
axes[1].set_title("Solo KDE (curva de densidad)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Rating")

# 3. Ambos juntos (la mejor vista)
sns.histplot(df["Rating"], kde=True, bins=20, color="#C82B40", ax=axes[2])
axes[2].set_title("Histograma + KDE (la mejor vista)", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Rating")

plt.tight_layout()
plt.show()

### 1.1 Formas de Distribución: ¿Por Qué Importa?

La **forma** de la distribución determina qué medidas son válidas y qué preguntas tienen sentido:

| Forma | Ejemplo en datos reales | Qué medida usar |
|-------|------------------------|----------------|
| **Simétrica** | Rating, estaturas, temperaturas | Media ≈ Mediana (cualquiera sirve) |
| **Sesgada a la derecha** | Salarios, montos de venta, tiempos de espera | **Mediana** (la media se infla) |
| **Sesgada a la izquierda** | Edad de jubilación, notas de examen fácil | **Mediana** |
| **Bimodal** | Mezcla de 2 poblaciones (hombre/mujer, 2 turnos) | Separar y analizar cada grupo |

> **Regla de oro:** Si la distribución es sesgada, la media **miente**. Usa la mediana.

In [ ]:
# Comparemos dos distribuciones de nuestro dataset: Rating vs Total
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating: ¿simétrica o sesgada?
sns.histplot(df["Rating"], kde=True, bins=25, color="#2563EB", alpha=0.6, ax=axes[0])
axes[0].axvline(df["Rating"].mean(), color="#C82B40", ls="--", lw=2, label=f'Media: {df["Rating"].mean():.2f}')
axes[0].axvline(df["Rating"].median(), color="#16A34A", ls="--", lw=2, label=f'Mediana: {df["Rating"].median():.2f}')
axes[0].legend(fontsize=10)
axes[0].set_title(f'Rating | skew = {df["Rating"].skew():.3f}', fontsize=13, fontweight="bold")

# Total: ¿simétrica o sesgada?
sns.histplot(df["Total"], kde=True, bins=25, color="#EA580C", alpha=0.6, ax=axes[1])
axes[1].axvline(df["Total"].mean(), color="#C82B40", ls="--", lw=2, label=f'Media: {df["Total"].mean():.1f}')
axes[1].axvline(df["Total"].median(), color="#16A34A", ls="--", lw=2, label=f'Mediana: {df["Total"].median():.1f}')
axes[1].legend(fontsize=10)
axes[1].set_title(f'Total | skew = {df["Total"].skew():.3f}', fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

print("Observa:")
print("- Rating: media y mediana están MUY cerca → distribución simétrica")
print("- Total: media > mediana → cola larga a la derecha (sesgo positivo)")

### 1.2 Sesgo (Skewness): Midiendo la Asimetría con un Número

El **sesgo** cuantifica cuánto y hacia dónde se desvía la distribución de la simetría:

$$g_1 = \frac{1}{n} \sum_{i=1}^{n} \left(\frac{x_i - \bar{x}}{s}\right)^{3}$$

**¿Cómo interpretar el resultado?**

| Valor de skew | Significado | Consecuencia práctica |
|--------------|-------------|----------------------|
| **-0.5 < skew < 0.5** | Aproximadamente simétrica | Media ≈ Mediana, ambas son confiables |
| **0.5 < \|skew\| < 1** | Sesgo moderado | Prefiere la mediana para reportar |
| **\|skew\| > 1** | Sesgo fuerte | La media **no** representa al grupo típico |

**¿Por qué al cubo?** Si eleváramos al cuadrado (como en la varianza), los signos se cancelarían. El cubo preserva si el dato está a la izquierda (−) o a la derecha (+) de la media.

En pandas: `df["col"].skew()`

In [ ]:
# Sesgo de todas las columnas numéricas
cols_num = df.select_dtypes(include="number").columns
print("=== Sesgo (skewness) por columna ===\n")
for col in cols_num:
    sk = df[col].skew()
    if abs(sk) < 0.5:
        juicio = "simétrica"
    elif abs(sk) < 1:
        juicio = "sesgo moderado"
    else:
        juicio = "sesgo FUERTE"
    print(f"  {col:>15}  skew = {sk:+.3f}  →  {juicio}")

---
## 2. Tendencia Central en Profundidad

### ¿Qué buscamos cuando calculamos el "centro"?

Cuando tenemos 1,000 transacciones, no podemos mirar una por una. Necesitamos **un solo número** que represente al grupo. Ese número es la **tendencia central**: un valor que resuma "lo típico".

Pero hay un problema: **no existe un solo "centro" correcto**. Depende de los datos y de la pregunta que hagamos.

### Media: "Si repartiera todo equitativamente, ¿cuánto le toca a cada uno?"

La media suma todos los valores y divide entre el total. Es como **repartir la torta en partes iguales**.

$$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i = \frac{x_1 + x_2 + \cdots + x_n}{n}$$

| Símbolo | Significado |
|---------|------------|
| $\bar{x}$ | La media (se lee "x barra"). Es el resultado. |
| $x_i$ | Cada dato individual ($x_1$ = primer dato, $x_2$ = segundo...). |
| $n$ | Cantidad total de datos. |
| $\sum_{i=1}^{n}$ | "Suma desde $i=1$ hasta $n$". Suma **todos** los datos. |

**Ejemplo:** Datos = {4, 8, 6, 10, 2}. $\bar{x} = \frac{4+8+6+10+2}{5} = \frac{30}{5} = 6$

**¿Qué información nos da?**
- Es útil para **presupuestos y proyecciones**: "si vendemos 1,000 unidades al mes, el ingreso promedio por unidad es $X, entonces esperamos $1000X de ingreso total".
- Funciona bien cuando los datos son **simétricos** (la campana es pareja), porque en ese caso la media cae justo en el centro.
- **Peligro:** Un solo dato extremo puede moverla mucho. Si 9 empleados ganan \$3,000 y 1 gana \$100,000, la media es \$12,700 — un número que **no representa a nadie**.

### Mediana: "¿Cuál es el valor del medio si ordeno todos los datos?"

La mediana ordena todos los datos de menor a mayor y toma el valor que queda **exactamente en la mitad**. El 50% de los datos está arriba y el 50% está abajo.

**¿Qué información nos da?**
- Nos dice cuál es la **experiencia típica real**. Si la mediana de venta es \$254, significa que la mitad de las transacciones están por debajo y la mitad por encima. Eso es "lo que le pasa a un cliente normal".
- Es **inmune a valores extremos**: si un cliente compra \$50,000, la mediana apenas se mueve.
- Es la medida correcta cuando hay **sesgo u outliers** (salarios, montos de venta, tiempos de espera, precios de vivienda).

### Moda: "¿Qué es lo más popular?"

La moda es el valor que **más se repite**. Es la única medida de centro que funciona con texto.

**¿Qué información nos da?**
- Responde preguntas de **preferencia o frecuencia**: "¿Cuál es el método de pago más usado?", "¿Cuál sucursal tiene más transacciones?"
- En variables numéricas discretas (cantidad de productos), te dice el valor más común.

### Resumen: ¿cuándo usar cada una?

| Medida | ¿Qué pregunta responde? | Uso práctico en Arca |
|--------|------------------------|---------------------|
| **Media** | "Si repartiera el total entre todos, ¿cuánto le toca a cada uno?" | Presupuestos, costo promedio por unidad, KPIs cuando los datos son **simétricos** |
| **Mediana** | "¿Cuál es la experiencia típica real?" | Montos de venta, tiempos de espera: cuando hay **sesgo u outliers** |
| **Moda** | "¿Qué es lo más popular/frecuente?" | Método de pago favorito, sucursal más activa, categoría top |

In [ ]:
# Comparación: media vs mediana en nuestras columnas numéricas
print("=== Media vs Mediana: ¿Cuál usar? ===\n")
print(f"{'Columna':>15}  {'Media':>10}  {'Mediana':>10}  {'Diferencia':>10}  {'Usar'}")
print("-" * 70)

for col in ["Total", "Rating", "gross income", "Tax 5%", "Quantity"]:
    media = df[col].mean()
    mediana = df[col].median()
    diff = abs(media - mediana)
    # Si la diferencia relativa es grande, preferir mediana
    diff_pct = diff / media * 100 if media != 0 else 0
    usar = "Cualquiera" if diff_pct < 5 else "MEDIANA"
    print(f"{col:>15}  {media:>10.2f}  {mediana:>10.2f}  {diff:>10.2f}  {usar}")

### 2.1 Cuándo la Media Miente: Un Ejemplo Claro

Imaginemos los salarios de 10 empleados en una empresa. 7 ganan entre \$2,000 y \$4,000, pero hay 3 ejecutivos que ganan mucho más. Veamos qué pasa con la media:

In [ ]:
# Ejemplo: la media miente cuando hay outliers
salarios = pd.Series([2000, 2500, 2800, 3000, 3200, 3500, 3800,
                       15000, 25000, 50000],
                      name="Salario ($)")

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(salarios, bins=15, kde=True, color="#2563EB", alpha=0.5, ax=ax)
ax.axvline(salarios.mean(), color="#C82B40", ls="--", lw=2.5,
           label=f"Media: ${salarios.mean():,.0f}")
ax.axvline(salarios.median(), color="#16A34A", ls="--", lw=2.5,
           label=f"Mediana: ${salarios.median():,.0f}")
ax.legend(fontsize=12)
ax.set_title("Salarios: 3 ejecutivos arrastran la media", fontsize=13, fontweight="bold")
ax.set_xlabel("Salario ($)", fontsize=11)
plt.tight_layout()
plt.show()

print(f"Media:   ${salarios.mean():>10,.0f}  ← ¿Alguien 'típico' gana esto? NO")
print(f"Mediana: ${salarios.median():>10,.0f}  ← ESTO representa al empleado típico")
print(f"\n7 de 10 empleados ganan MENOS que la media.")
print("La media 'miente' porque 3 salarios extremos la inflan.")

### 2.2 Árbol de Decisión: ¿Qué Medida Uso?

```
¿La variable es numérica?
│
├── SÍ → ¿La distribución es simétrica? (skew ≈ 0, media ≈ mediana)
│        │
│        ├── SÍ → usa la MEDIA
│        │        ¿Por qué? Usa TODOS los datos, es más precisa.
│        │        Da información de: presupuesto, proyección, KPIs.
│        │        Ejemplo: "El ingreso promedio por transacción es $322"
│        │        → puedo multiplicar por N transacciones para proyectar.
│        │
│        └── NO (sesgada u outliers) → usa la MEDIANA
│                 ¿Por qué? No se deja arrastrar por extremos.
│                 Da información de: la experiencia TÍPICA real.
│                 Ejemplo: "La venta típica es $254" → eso es lo que
│                 vive un cliente normal, no un número inflado.
│
└── NO (es texto/categoría) → usa la MODA
         ¿Por qué? Es la única que funciona con texto.
         Da información de: preferencia, popularidad.
         Ejemplo: "El método de pago más usado es Ewallet"
         → para decidir qué métodos priorizar.
```

**Regla práctica:** Siempre calcula **ambas** (media y mediana). Si difieren mucho, hay sesgo u outliers, y la mediana es más confiable para comunicar el "típico". La media sigue siendo útil para calcular totales y presupuestos.

In [ ]:
# Tendencia central en nuestro dataset: ahora con CONTEXTO de negocio
print("=" * 65)
print("  REPORTE: Tendencia Central — Supermarket Sales")
print("=" * 65)

# Numéricas: decidir media o mediana según simetría
for col in ["Total", "Rating", "Quantity"]:
    media = df[col].mean()
    mediana = df[col].median()
    sk = df[col].skew()
    valor_central = mediana if abs(sk) > 0.5 else media
    medida = "mediana" if abs(sk) > 0.5 else "media"
    print(f"\n  {col}:")
    print(f"    Media = {media:.2f} | Mediana = {mediana:.2f} | Skew = {sk:.3f}")
    print(f"    → Valor típico (usando {medida}): {valor_central:.2f}")

# Categóricas: moda
print()
for col in ["Payment", "Branch", "Product line"]:
    moda = df[col].mode()[0]
    freq = df[col].value_counts().iloc[0]
    pct = freq / len(df) * 100
    print(f"  {col}:")
    print(f"    Moda = '{moda}' ({freq} veces, {pct:.1f}%)")

---
## 3. Medidas de Dispersión: ¿Qué Tan Dispersos Están los Datos?

### La tendencia central NO es suficiente

Dos grupos pueden tener **la misma media** pero realidades **totalmente distintas**:

| | Grupo A (salarios) | Grupo B (salarios) |
|--|---|---|
| Valores | \$3000, \$3100, \$2900, \$3050, \$2950 | \$500, \$1000, \$3000, \$5000, \$5500 |
| **Media** | **\$3,000** | **\$3,000** |
| Realidad | Todos ganan parecido | Enorme desigualdad |

La **dispersión** completa la historia que la media no cuenta. Mide **qué tan separados están los datos entre sí**.

---

### 3.1 Rango: La medida más simple

$$\text{Rango} = \text{máximo} - \text{mínimo}$$

Solo mira los dos extremos. Es fácil de entender, pero **frágil**: un solo outlier lo infla enormemente. Si 999 ventas están entre \$10 y \$500 pero hay una de \$10,000, el rango es \$9,990 — y eso no describe bien la realidad del 99.9% de los datos.

---

### 3.2 Cuartiles e IQR: Midiendo el "corazón" de los datos

#### ¿Qué son los cuartiles?

Imagina que ordenas **todos** los datos de menor a mayor y los divides en **4 partes iguales** (como cortar un pastel en 4). Los puntos donde cortas se llaman **cuartiles**:

```
 Datos ordenados de menor a mayor:
 ┌──────────┬──────────┬──────────┬──────────┐
 │  25% más │  Siguiente│ Siguiente │  25% más  │
 │  bajos   │  25%      │  25%      │  altos    │
 └──────────┴──────────┴──────────┴──────────┘
           Q1         Q2          Q3
        (percentil  (percentil  (percentil
          25)         50)         75)
```

| Cuartil | Significado | Interpretación |
|---------|------------|---------------|
| **Q1** (percentil 25) | El 25% de los datos está **por debajo** de este valor | "Las ventas bajas llegan hasta aquí" |
| **Q2** (percentil 50) | El 50% está por debajo = **es la mediana** | "La venta típica" |
| **Q3** (percentil 75) | El 75% de los datos está **por debajo** de este valor | "Solo el 25% de las ventas supera esto" |

#### ¿Cómo se calcula un cuartil?

**Ejemplo paso a paso** con 8 datos: {12, 15, 18, 22, 25, 30, 35, 50}

1. **Ordénalos** de menor a mayor (ya están ordenados).
2. **Q2 (mediana):** el valor del medio. Con 8 datos, es el promedio del 4to y 5to: $(22 + 25) / 2 = 23.5$
3. **Q1:** la mediana de la **mitad inferior** {12, 15, 18, 22}: $(15 + 18) / 2 = 16.5$
4. **Q3:** la mediana de la **mitad superior** {25, 30, 35, 50}: $(30 + 35) / 2 = 32.5$

En pandas: `df["col"].quantile(0.25)` para Q1, `.quantile(0.75)` para Q3.

#### ¿Qué es el IQR?

$$\text{IQR} = Q3 - Q1$$

El IQR (Rango Intercuartílico) mide **qué tan ancho es el rango del 50% central** de los datos. Ignora completamente el 25% más bajo y el 25% más alto.

**En el ejemplo:** IQR = 32.5 − 16.5 = 16

**¿Por qué es útil?**
- Es **robusta a outliers**: como solo mira el 50% central, un dato extremo no la afecta.
- Complementa a la mediana: si usas la mediana como centro, usa el IQR como dispersión.
- Es la base del **método IQR para detectar outliers** (sección 4).

**¿Qué información nos da?**
- "El 50% central de las ventas está entre \$Q1 y \$Q3, un rango de \$IQR."
- Un IQR pequeño → los datos están concentrados en el centro.
- Un IQR grande → los datos están muy dispersos.

---

### Resumen de medidas de dispersión

| Medida | Fórmula | Interpretación | Código | Robusta a outliers? |
|--------|---------|---------------|--------|-------------------|
| **Rango** | max − min | Diferencia entre extremos | `.max() - .min()` | No |
| **Varianza** ($s^2$) | $\frac{\sum(x_i - \bar{x})^2}{n-1}$ | Dispersión promedio (en unidades²) | `.var()` | No |
| **Desv. estándar** ($s$) | $\sqrt{s^2}$ | Dispersión en las **mismas unidades** que los datos | `.std()` | No |
| **IQR** | Q3 − Q1 | Ancho del 50% central | `.quantile(.75) - .quantile(.25)` | **Sí** |
| **CV** | $\frac{s}{\|\bar{x}\|} \times 100\%$ | Dispersión **relativa** (%) | `std / mean * 100` | No |

In [ ]:
# Cuartiles e IQR paso a paso con un ejemplo pequeño
ejemplo = pd.Series([12, 15, 18, 22, 25, 30, 35, 50])
print("Datos:", ejemplo.tolist())
print(f"\nQ1 (percentil 25): {ejemplo.quantile(0.25)}")
print(f"Q2 (mediana):       {ejemplo.quantile(0.50)}")
print(f"Q3 (percentil 75): {ejemplo.quantile(0.75)}")
print(f"IQR = Q3 - Q1:     {ejemplo.quantile(0.75) - ejemplo.quantile(0.25)}")
print(f"\nInterpretación: el 50% central de los datos está entre")
print(f"{ejemplo.quantile(0.25)} y {ejemplo.quantile(0.75)}.")

### 3.3 Varianza y Desviación Estándar

#### ¿Qué queremos medir?

Queremos saber: **en promedio, qué tan lejos está cada dato de la media**. Si todos están cerca de la media, hay poca dispersión. Si están lejos, hay mucha.

#### La fórmula

$$s^2 = \frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2 \qquad\qquad s = \sqrt{s^2}$$

| Paso | Qué hacemos | Por qué |
|------|------------|---------|
| $x_i - \bar{x}$ | Calculamos la distancia de cada dato a la media | Para saber cuánto se aleja cada uno |
| $(x_i - \bar{x})^2$ | Elevamos al cuadrado | Para que todas las distancias sean **positivas** (si no, las negativas cancelarían a las positivas y la suma daría ≈ 0) |
| $\sum$ | Sumamos todas las distancias² | Para acumular cuánta dispersión hay en total |
| $\div (n-1)$ | Dividimos entre $n-1$ | Para obtener un **promedio** de dispersión (se usa $n-1$ en vez de $n$ para corregir sesgo en muestras) |
| $\sqrt{\phantom{x}}$ | Sacamos la raíz cuadrada | Para volver a las **unidades originales** (de dólares² a dólares) |

**Ejemplo paso a paso:** Datos = {2, 4, 4, 4, 5, 5, 7, 9}. Media $\bar{x} = 5$.

Distancias²: $(2-5)^2 + (4-5)^2 + (4-5)^2 + (4-5)^2 + (5-5)^2 + (5-5)^2 + (7-5)^2 + (9-5)^2$
$= 9 + 1 + 1 + 1 + 0 + 0 + 4 + 16 = 32$

$s^2 = \frac{32}{7} = 4.57$ (varianza) → $s = \sqrt{4.57} = 2.14$ (desviación estándar)

**Interpretación:** "En promedio, los datos se alejan **2.14 unidades** de la media."

#### ¿Cuándo usar varianza vs desviación estándar?

| Medida | ¿Qué información da? | ¿Cuándo usarla? |
|--------|----------------------|-----------------|
| **Varianza** ($s^2$) | Dispersión en unidades² (ej: dólares²). Es difícil de interpretar "en la cabeza" | En **fórmulas** y modelos matemáticos (regresión, ANOVA). Para **reportar al negocio**, casi nunca |
| **Desv. estándar** ($s$) | "En promedio, los datos se alejan $s$ unidades de la media." Misma escala que los datos → **interpretable** | Para **comunicar** dispersión a personas. En reportes, gráficos, Z-score |

> **Regla simple:** Comunica con `.std()`; usa `.var()` solo cuando el código o la teoría lo pida.

> **Cuidado:** Tanto la std como la varianza son **sensibles a outliers** (como la media). Si la distribución tiene sesgo fuerte, complementa con el **IQR**.

In [ ]:
# Todas las medidas de dispersión para "Total" — con interpretación
col = "Total"
data = df[col].dropna()

rango    = data.max() - data.min()
varianza = data.var()
std      = data.std()
q1       = data.quantile(0.25)
q3       = data.quantile(0.75)
iqr      = q3 - q1

print(f"=== Dispersión de '{col}' ===\n")
print(f"  Rango:       {rango:>10.2f}  (max - min: fácil pero frágil ante outliers)")
print(f"  Varianza:    {varianza:>10.2f}  (en dólares² — útil en fórmulas, no para reportar)")
print(f"  Std:         {std:>10.2f}  (en dólares — para comunicar dispersión)")
print(f"  Q1:          {q1:>10.2f}  (el 25% de ventas está por debajo de esto)")
print(f"  Q3:          {q3:>10.2f}  (el 75% de ventas está por debajo de esto)")
print(f"  IQR:         {iqr:>10.2f}  (ancho del 50% central — robusta a outliers)")

print(f"\n--- ¿Qué le digo al gerente? ---")
print(f"  'La venta típica (mediana) es ${data.median():.0f}.'")
print(f"  'El 50% central de las ventas va de ${q1:.0f} a ${q3:.0f}.'")
print(f"  'Las ventas suelen alejarse ${std:.0f} dólares del promedio.'")
print(f"  'El rango total va de ${data.min():.0f} a ${data.max():.0f},")
print(f"   pero ese rango está inflado por compras extremas.'")

### 3.2 `df.describe()` — El Resumen Completo en Una Línea

`describe()` nos da todo de una vez: count, mean, std, min, 25%, 50% (mediana), 75%, max.

In [ ]:
# describe() nos da el resumen completo
df[["Total", "Rating", "Quantity"]].describe().round(2)

### 3.3 Coeficiente de Variación (CV): Comparar Manzanas con Naranjas

La std compara dispersión cuando la **unidad es la misma**. Pero ¿cómo comparamos la dispersión de `Rating` (escala 1-10) con `Total` (dólares)?

$$\text{CV} = \frac{s}{|\bar{x}|} \times 100\%$$

El CV responde: "¿Qué tan grande es la desviación estándar *respecto* a la media?"

**Cuándo usar CV:**
- Comparar **dos columnas con unidades distintas** (Rating vs Total)
- Comparar **la misma métrica entre grupos** ("¿En qué sucursal las ventas oscilan más?")

**Cuándo NO usar CV:**
- Si la media es ≈ 0 o hay valores negativos mezclados (el CV se vuelve engañoso)

> CV alto (> 30-40%) suele indicar mucha variabilidad **respecto al nivel típico**.

In [ ]:
# CV: ¿Qué variable es más heterogénea en términos relativos?
print("=== Coeficiente de Variación (CV) ===\n")
for col in ["Total", "Rating", "Quantity", "gross income"]:
    media = df[col].mean()
    std = df[col].std()
    cv = (std / abs(media)) * 100
    nivel = "ALTA variabilidad" if cv > 40 else "Moderada" if cv > 20 else "Baja"
    print(f"  {col:>15}  media={media:>8.2f}  std={std:>8.2f}  CV={cv:>6.1f}%  → {nivel}")

print("\n  Lectura: aunque la std de Total ($246) es enorme comparada")
print("  con la de Rating (1.7), el CV nos dice cuál es más variable")
print("  EN PROPORCIÓN a su promedio.")

# CV por sucursal: ¿dónde son más impredecibles las ventas?
print("\n=== CV de Total por Sucursal ===\n")
for branch in sorted(df["Branch"].unique()):
    sub = df[df["Branch"] == branch]["Total"]
    cv = (sub.std() / sub.mean()) * 100
    print(f"  Sucursal {branch}: CV = {cv:.1f}%")

---
## 4. Outliers: Detectar y Decidir Qué Hacer

### ¿Qué es un outlier?

Un **outlier** es un dato que se aleja significativamente del resto de observaciones.

**Importante:** Un outlier no siempre es un error.
- Una compra de \$1,000 puede ser un cliente que compró mucho → **dato legítimo**
- Una edad de −5 años o un salario de \$0 → **error claro**

> **Nunca elimines outliers automáticamente.** Primero entiende *por qué* el dato es extremo. El contexto de negocio decide.

### Dos métodos para detectarlos

| Método | Cómo funciona | Cuándo usarlo |
|--------|--------------|--------------|
| **IQR** | Fuera de [Q1 − 1.5×IQR, Q3 + 1.5×IQR] | Siempre funciona. No asume normalidad |
| **Z-score** | \|z\| > 2 o 3 | Mejor si la distribución es aprox. normal |

In [ ]:
# Método 1: IQR
col = "Total"
data = df[col].dropna()

q1 = data.quantile(0.25)
q3 = data.quantile(0.75)
iqr = q3 - q1

lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr

outliers_iqr = data[(data < lim_inf) | (data > lim_sup)]

print("=== Método IQR ===")
print(f"  Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}")
print(f"  Límite inferior: {lim_inf:.2f}")
print(f"  Límite superior: {lim_sup:.2f}")
print(f"  Outliers encontrados: {len(outliers_iqr)} ({len(outliers_iqr)/len(data)*100:.1f}%)")

# Método 2: Z-score
z_scores = (data - data.mean()) / data.std()
outliers_z2 = data[z_scores.abs() > 2]
outliers_z3 = data[z_scores.abs() > 3]

print(f"\n=== Método Z-score ===")
print(f"  Datos con |z| > 2: {len(outliers_z2)} ({len(outliers_z2)/len(data)*100:.1f}%)")
print(f"  Datos con |z| > 3: {len(outliers_z3)} ({len(outliers_z3)/len(data)*100:.1f}%)")

print(f"\n  Nota: Los dos métodos no siempre coinciden.")
print(f"  IQR no asume normalidad (más conservador).")
print(f"  Z-score funciona mejor si la distribución es simétrica.")

In [ ]:
# Visualizar outliers: Boxplot + puntos reales (stripplot)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot horizontal de Total
sns.boxplot(data=df, x="Total", color="#C82B40", width=0.3, ax=axes[0])
sns.stripplot(data=df, x="Total", color="#2563EB", alpha=0.3, size=3, ax=axes[0])
axes[0].set_title("Total: Boxplot + puntos reales", fontsize=13, fontweight="bold")

# Boxplot de Total por Branch: ¿alguna sucursal tiene más outliers?
sns.boxplot(data=df, x="Branch", y="Total", palette=["#C82B40", "#2563EB", "#16A34A"], ax=axes[1])
axes[1].set_title("Total por Sucursal", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

### 4.1 ¿Eliminar o Conservar? Criterio de Negocio

| Eliminar cuando... | Conservar cuando... |
|--------------------|---------------------|
| Es un **error de registro** (edad negativa, precio \$0) | Es un dato **real pero extremo** (cliente VIP, compra navideña) |
| **Distorsiona el modelo** (regresión, clustering) | Es justamente lo que **buscas** (detección de fraude) |
| El porcentaje es **< 1-2%** y no afecta representatividad | Representa un **segmento real** que el negocio debe atender |

> **Alternativa:** En lugar de eliminar, puedes **truncar** (winsorize) los valores extremos al percentil 1% y 99%.

In [ ]:
# Impacto de los outliers: ¿cuánto cambian las estadísticas si los eliminamos?
data_total = df["Total"].dropna()

# Sin outliers (método IQR)
q1 = data_total.quantile(0.25)
q3 = data_total.quantile(0.75)
iqr = q3 - q1
mask = (data_total >= q1 - 1.5 * iqr) & (data_total <= q3 + 1.5 * iqr)
data_sin_outliers = data_total[mask]

print("=== Impacto de Outliers en las Estadísticas ===\n")
print(f"{'Medida':<15} {'Con outliers':>15} {'Sin outliers':>15} {'Cambio':>10}")
print("-" * 60)

for nombre, func in [("Media", "mean"), ("Mediana", "median"), ("Std", "std")]:
    con = getattr(data_total, func)()
    sin = getattr(data_sin_outliers, func)()
    cambio = (sin - con) / con * 100
    print(f"{nombre:<15} {con:>15.2f} {sin:>15.2f} {cambio:>+9.1f}%")

print(f"{'N filas':<15} {len(data_total):>15d} {len(data_sin_outliers):>15d} "
      f"{-(len(data_total)-len(data_sin_outliers)):>+9d}")
print(f"\nObserva: la MEDIANA apenas cambia → es robusta a outliers.")

---
## 5. La Distribución Normal y la Regla 68-95-99.7

### ¿Por qué importa la distribución Normal?

La distribución Normal (o Gaussiana, o "campana de Gauss") es la distribución más importante en estadística por tres razones:

1. **Aparece en la naturaleza:** estaturas, pesos, errores de medición, temperaturas...
2. **Base matemática:** Muchos tests estadísticos y modelos de ML asumen datos normales.
3. **Teorema Central del Límite:** Los promedios de muestras tienden a ser normales, sin importar la distribución original.

Se define con solo dos parámetros: la **media** ($\mu$) y la **desviación estándar** ($\sigma$).

### La Regla 68-95-99.7 (Regla Empírica)

Si los datos son normales:
- **68%** de los datos caen dentro de 1 desviación estándar de la media ($\mu \pm \sigma$)
- **95%** caen dentro de 2 desviaciones estándar ($\mu \pm 2\sigma$)
- **99.7%** caen dentro de 3 desviaciones estándar ($\mu \pm 3\sigma$)

Esto significa que un dato a más de 3$\sigma$ de la media es **extremadamente raro** (solo 0.3% de probabilidad).

In [ ]:
# Verificar la regla 68-95-99.7 en Rating (que sospechamos es ~Normal)
data_r = df["Rating"]
mu = data_r.mean()
sigma = data_r.std()

dentro_1s = ((data_r >= mu - sigma) & (data_r <= mu + sigma)).mean() * 100
dentro_2s = ((data_r >= mu - 2*sigma) & (data_r <= mu + 2*sigma)).mean() * 100
dentro_3s = ((data_r >= mu - 3*sigma) & (data_r <= mu + 3*sigma)).mean() * 100

print("=== Regla 68-95-99.7 aplicada a Rating ===\n")
print(f"  Media (μ) = {mu:.2f}, Std (σ) = {sigma:.2f}\n")
print(f"  {'Rango':<25} {'Esperado':>10} {'Observado':>10}")
print(f"  {'-'*48}")
print(f"  {'μ ± 1σ (' + f'{mu-sigma:.1f} a {mu+sigma:.1f}' + ')':<25} {'68.0%':>10} {dentro_1s:>9.1f}%")
print(f"  {'μ ± 2σ (' + f'{mu-2*sigma:.1f} a {mu+2*sigma:.1f}' + ')':<25} {'95.0%':>10} {dentro_2s:>9.1f}%")
print(f"  {'μ ± 3σ (' + f'{mu-3*sigma:.1f} a {mu+3*sigma:.1f}' + ')':<25} {'99.7%':>10} {dentro_3s:>9.1f}%")

# Visualizar la regla
fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(data_r, kde=True, bins=25, color="#2563EB", alpha=0.4, stat="density", ax=ax)

# Dibujar las bandas
colors = ["#16A34A", "#EA580C", "#C82B40"]
alphas = [0.15, 0.10, 0.05]
labels = [f"68% (μ±1σ)", f"95% (μ±2σ)", f"99.7% (μ±3σ)"]
for i, (c, a, l) in enumerate(zip(colors, alphas, labels), 1):
    ax.axvspan(mu - i*sigma, mu + i*sigma, alpha=a, color=c, label=l)

ax.axvline(mu, color="#C82B40", ls="--", lw=2, label=f"Media: {mu:.2f}")
ax.legend(fontsize=10)
ax.set_title("Regla 68-95-99.7 en Rating", fontsize=13, fontweight="bold")
ax.set_xlabel("Rating", fontsize=11)
plt.tight_layout()
plt.show()

### 5.1 ¿Cómo Saber si los Datos Son Normales?

Tres herramientas, de la más simple a la más rigurosa:

| # | Herramienta | Qué hace | Veredicto |
|---|------------|----------|-----------|
| 1 | **Histograma + KDE** | Inspección visual: ¿se ve simétrica? ¿un solo pico? | Rápido pero subjetivo |
| 2 | **Q-Q Plot** | Compara tus datos con una Normal teórica. Si los puntos siguen la diagonal → Normal | Visual pero más preciso |
| 3 | **Test de Shapiro-Wilk** | Devuelve un **p-value**: p > 0.05 → Normal, p ≤ 0.05 → No Normal | Numérico y formal |

> Lo ideal es usar las tres juntas. Lo visual da intuición, el Q-Q plot da detalle, y el test da una respuesta numérica.

In [ ]:
# Las 3 herramientas para verificar normalidad: Rating vs Total
from scipy.stats import shapiro

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for row, col_name, color in [(0, "Rating", "#2563EB"), (1, "Total", "#EA580C")]:
    data = df[col_name].dropna()

    # 1. Histograma + KDE
    sns.histplot(data, kde=True, bins=25, color=color, alpha=0.5, ax=axes[row, 0])
    axes[row, 0].set_title(f"{col_name}: Histograma + KDE\nskew = {data.skew():.3f}",
                           fontsize=11, fontweight="bold")

    # 2. Q-Q Plot
    stats.probplot(data, dist="norm", plot=axes[row, 1])
    axes[row, 1].set_title(f"{col_name}: Q-Q Plot", fontsize=11, fontweight="bold")
    axes[row, 1].get_lines()[0].set_color(color)
    axes[row, 1].get_lines()[0].set_markersize(3)

    # 3. Shapiro-Wilk (mostrar resultado como texto en el gráfico)
    stat_val, p_val = shapiro(data.sample(min(500, len(data)), random_state=42))
    resultado = "Normal" if p_val > 0.05 else "NO Normal"
    color_res = "#16A34A" if p_val > 0.05 else "#C82B40"

    axes[row, 2].text(0.5, 0.6, f"Test de Shapiro-Wilk",
                      ha="center", va="center", fontsize=14, fontweight="bold",
                      transform=axes[row, 2].transAxes)
    axes[row, 2].text(0.5, 0.45, f"stat = {stat_val:.4f}",
                      ha="center", va="center", fontsize=12,
                      transform=axes[row, 2].transAxes)
    axes[row, 2].text(0.5, 0.32, f"p-value = {p_val:.4f}",
                      ha="center", va="center", fontsize=12,
                      transform=axes[row, 2].transAxes)
    axes[row, 2].text(0.5, 0.15, f"Veredicto: {resultado}",
                      ha="center", va="center", fontsize=16, fontweight="bold",
                      color=color_res, transform=axes[row, 2].transAxes)
    axes[row, 2].set_title(f"{col_name}: Shapiro-Wilk", fontsize=11, fontweight="bold")
    axes[row, 2].set_xticks([])
    axes[row, 2].set_yticks([])

plt.tight_layout()
plt.show()

print("Interpretación del p-value:")
print("  p > 0.05: 'No hay suficiente evidencia para decir que NO es Normal'")
print("  p ≤ 0.05: 'Hay evidencia estadística de que NO es Normal'")

### 5.2 Z-Score: ¿Qué Tan Raro Es un Dato?

El Z-score mide cuántas desviaciones estándar separan un dato de la media:

$$z = \frac{x - \mu}{\sigma}$$

| Z-score | Interpretación |
|---------|---------------|
| \|z\| < 2 | Dato **normal** (dentro de lo esperado) |
| \|z\| > 2 | Dato **inusual** (merece investigación) |
| \|z\| > 3 | **Outlier fuerte** (solo 0.3% en una Normal perfecta) |

El Z-score es útil porque **normaliza**: convierte cualquier variable a una escala estándar donde 0 es la media y 1 es una desviación estándar.

In [ ]:
# Z-score en acción: ¿qué tan raras son las ventas más altas?
data = df["Total"].dropna()
z_scores = (data - data.mean()) / data.std()

# Agregar z-score al dataframe para inspeccionar
df_z = df[["Branch", "Product line", "Total"]].copy()
df_z["z_score"] = (df["Total"] - df["Total"].mean()) / df["Total"].std()

print("=== Las 10 ventas más extremas (mayor |z-score|) ===\n")
top_extremos = df_z.reindex(df_z["z_score"].abs().sort_values(ascending=False).index).head(10)
print(top_extremos.to_string(index=False))

print(f"\n--- Resumen Z-score de Total ---")
print(f"  Datos con |z| > 2:  {(z_scores.abs() > 2).sum()} ({(z_scores.abs() > 2).mean()*100:.1f}%)")
print(f"  Datos con |z| > 3:  {(z_scores.abs() > 3).sum()} ({(z_scores.abs() > 3).mean()*100:.1f}%)")
print(f"\n  En una Normal perfecta esperaríamos ~5% con |z|>2 y ~0.3% con |z|>3")

### 5.3 ¿Normal o No? — Resumen de Decisión y Consecuencias

| Herramienta | Rating | Total |
|------------|--------|-------|
| Histograma + KDE | Simétrica | Sesgada |
| Skewness | ≈ 0 | > 0.5 |
| Q-Q Plot | Puntos en la línea | Se curvan |
| Shapiro-Wilk | p > 0.05 | p ≤ 0.05 |
| **Veredicto** | **≈ Normal** | **No Normal** |

### ¿Y si NO es Normal? Consecuencias prácticas

| Si es Normal... | Si NO es Normal... |
|----------------|-------------------|
| Usa la **media** como centro | Usa la **mediana** |
| Usa la **std** como dispersión | Usa el **IQR** |
| Z-score funciona bien para outliers | Prefiere el método **IQR** |
| Puedes aplicar tests paramétricos | Necesitas tests no paramétricos o transformar |

> En la vida real, pocos datos son perfectamente normales. Lo importante es saber **qué tan lejos están** y ajustar las herramientas.

---
## 6. De Estadísticas a Información: Convertir Números en Decisiones

### El paso que falta: DIKW en acción

Recordemos la pirámide DIKW de la Clase 8:

| Nivel | En esta clase |
|-------|--------------|
| **Data** | Los 1,000 registros de ventas del supermercado |
| **Information** | Las estadísticas que calculamos (media, mediana, skew, CV, outliers...) |
| **Knowledge** | Interpretar qué significan para el negocio |
| **Wisdom** | Tomar decisiones y actuar |

Los números por sí solos son **datos**. Convertirlos en **información** requiere contexto, comparación y preguntarse: *"¿Y entonces qué?"*

### La pregunta clave: "¿Y entonces qué?"

Cada estadística debe responder una pregunta de negocio:

| Estadística que calculamos | Pregunta de negocio que responde |
|---------------------------|--------------------------------|
| Mediana de Total = \$254 | "¿Cuál es la venta típica?" → Para fijar metas de ventas realistas |
| CV de Total por sucursal | "¿Dónde son más impredecibles las ventas?" → Para priorizar dónde estabilizar |
| Outliers en Total | "¿Hay ventas inusualmente altas?" → ¿Son clientes VIP? ¿Errores? |
| Rating es Normal con media 7.0 | "¿Los clientes están satisfechos de manera consistente?" → Sí, con poca variación |
| Skew de Total > 0 | "¿La mayoría de ventas son pequeñas con pocas grandes?" → Ajustar promociones |

In [ ]:
# Ejemplo: Reporte ejecutivo generado automáticamente con estadísticas
print("=" * 65)
print("  REPORTE EJECUTIVO: Supermarket Sales — Insights")
print("=" * 65)

# 1. Venta típica
mediana_total = df["Total"].median()
media_total = df["Total"].mean()
print(f"\n1. VENTA TÍPICA")
print(f"   La venta mediana es ${mediana_total:.0f}.")
print(f"   (La media es ${media_total:.0f}, pero está inflada por ventas grandes.)")
print(f"   → Recomendación: usar ${mediana_total:.0f} como referencia para metas diarias.\n")

# 2. Dispersión por sucursal
print(f"2. ESTABILIDAD POR SUCURSAL")
for branch in sorted(df["Branch"].unique()):
    sub = df[df["Branch"] == branch]["Total"]
    cv = (sub.std() / sub.mean()) * 100
    mediana = sub.median()
    print(f"   Sucursal {branch}: mediana ${mediana:.0f}, CV = {cv:.0f}%", end="")
    if cv > 80:
        print(" ← ALTA variabilidad, investigar")
    else:
        print(" ← Estable")

# 3. Satisfacción
print(f"\n3. SATISFACCIÓN DEL CLIENTE")
media_r = df["Rating"].mean()
std_r = df["Rating"].std()
print(f"   Rating promedio: {media_r:.1f}/10 (std = {std_r:.1f})")
print(f"   Distribución simétrica → la satisfacción es consistente.")
print(f"   {((df['Rating'] >= 8).mean() * 100):.0f}% de clientes dan rating ≥ 8.")

# 4. Outliers
q1 = df["Total"].quantile(0.25)
q3 = df["Total"].quantile(0.75)
iqr = q3 - q1
n_outliers = ((df["Total"] < q1 - 1.5*iqr) | (df["Total"] > q3 + 1.5*iqr)).sum()
print(f"\n4. VENTAS ATÍPICAS")
print(f"   {n_outliers} transacciones ({n_outliers/len(df)*100:.1f}%) son outliers por IQR.")
print(f"   → Revisar si son clientes VIP o errores de registro.")

# 5. Producto estrella
top_product = df["Product line"].value_counts().index[0]
top_count = df["Product line"].value_counts().iloc[0]
top_revenue = df.groupby("Product line")["Total"].sum().sort_values(ascending=False)
print(f"\n5. PRODUCTO ESTRELLA")
print(f"   Más vendido: '{top_product}' ({top_count} transacciones)")
print(f"   Mayor ingreso: '{top_revenue.index[0]}' (${top_revenue.iloc[0]:,.0f})")

print("\n" + "=" * 65)

### Guía Rápida: De Estadística a Información

```
PASO 1: Mira la forma (histograma + KDE, skew)
   → ¿Simétrica o sesgada? Esto decide qué medidas usar.

PASO 2: Elige el "centro" correcto
   → Simétrica → media.  Sesgada → mediana.  Texto → moda.

PASO 3: Mide la dispersión
   → Dentro del mismo grupo → std o IQR
   → Comparar grupos/variables → CV

PASO 4: Busca outliers
   → IQR (siempre funciona) o Z-score (si ~Normal)
   → Pregunta: ¿error o dato legítimo?

PASO 5: Pregunta "¿Y entonces qué?"
   → Cada número debe responder una pregunta de negocio.
   → Si no puedes convertirlo en una acción, no es información útil.
```

---
## 7. Práctica Integradora

### Ejercicio 1: Explora la Distribución (5 min)

Sobre el dataset `df`:

1. Haz un histograma + KDE de `gross income`
2. Calcula su media, mediana y skew
3. ¿Es simétrica o sesgada? ¿Cuál medida de centro usarías?
4. Calcula el CV de `gross income` y compáralo con el de `Rating`

In [ ]:
# TODO: Resuelve el Ejercicio 1 aquí



<details>
<summary>Click para ver la solución del Ejercicio 1</summary>

```python
# 1. Histograma + KDE
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df["gross income"], kde=True, bins=25, color="#7C3AED", alpha=0.5, ax=ax)

# 2. Media, mediana y skew
media = df["gross income"].mean()
mediana = df["gross income"].median()
sk = df["gross income"].skew()

ax.axvline(media, color="#C82B40", ls="--", lw=2, label=f"Media: {media:.2f}")
ax.axvline(mediana, color="#16A34A", ls="--", lw=2, label=f"Mediana: {mediana:.2f}")
ax.legend()
ax.set_title(f"gross income | skew = {sk:.3f}")
plt.show()

# 3. Decisión
print(f"Media: {media:.2f}, Mediana: {mediana:.2f}, Skew: {sk:.3f}")
if abs(sk) < 0.5:
    print("→ Distribución simétrica, se puede usar la media.")
else:
    print("→ Distribución sesgada, usar la mediana.")

# 4. CV
cv_gi = (df["gross income"].std() / df["gross income"].mean()) * 100
cv_r = (df["Rating"].std() / df["Rating"].mean()) * 100
print(f"\nCV gross income: {cv_gi:.1f}%")
print(f"CV Rating: {cv_r:.1f}%")
print(f"→ gross income es más variable en proporción a su promedio.")
```
</details>

### Ejercicio 2: Detecta Outliers (5 min)

Aplica el método IQR y el Z-score a la columna `Total`:

1. Calcula Q1, Q3 e IQR
2. Define los límites inferior y superior
3. ¿Cuántos outliers hay con el método IQR?
4. ¿Cuántos datos tienen |z-score| > 2?
5. Crea un boxplot de `Total` por `Branch`. ¿Hay alguna sucursal con más outliers?
6. **Pregunta de negocio:** ¿Son errores o compras legítimamente grandes? ¿Qué harías con ellos?

In [ ]:
# TODO: Resuelve el Ejercicio 2 aquí



<details>
<summary>Click para ver la solución del Ejercicio 2</summary>

```python
col = "Total"
data = df[col].dropna()

# 1-2. Q1, Q3, IQR y límites
q1 = data.quantile(0.25)
q3 = data.quantile(0.75)
iqr = q3 - q1
lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr
print(f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}")
print(f"Límites: [{lim_inf:.2f}, {lim_sup:.2f}]")

# 3. Outliers IQR
outliers = data[(data < lim_inf) | (data > lim_sup)]
print(f"\nOutliers IQR: {len(outliers)} ({len(outliers)/len(data)*100:.1f}%)")

# 4. Z-score > 2
z = (data - data.mean()) / data.std()
print(f"Datos con |z| > 2: {(z.abs() > 2).sum()}")

# 5. Boxplot por Branch
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="Branch", y="Total",
            palette=["#C82B40", "#2563EB", "#16A34A"], ax=ax)
ax.set_title("Total por Sucursal — ¿Dónde hay más outliers?")
plt.show()

# 6. Pregunta de negocio
print("Los outliers son ventas altas ($500+). Son compras")
print("legítimas de clientes que compraron muchos productos.")
print("Recomendación: CONSERVAR y estudiar qué compraron.")
```
</details>

### Ejercicio 3: Pipeline Completo — De Datos a Información (15 min)

Realiza un análisis estadístico completo de la columna `Quantity` (cantidad de productos por transacción):

1. **Distribución:** Histograma + KDE. ¿Qué forma tiene?
2. **Centro:** Calcula media, mediana y moda. ¿Cuál usarías para reportar?
3. **Dispersión:** Calcula std, IQR y CV. ¿Es muy dispersa?
4. **Outliers:** Aplica IQR. ¿Hay outliers?
5. **Normalidad:** Aplica las 3 herramientas (visual, Q-Q, Shapiro). ¿Es Normal?
6. **Información:** Escribe 2-3 oraciones que conviertan tus hallazgos en **información para el gerente del supermercado**. Ejemplo: "La mayoría de clientes compran entre X y Y productos por visita..."

In [ ]:
# TODO: Resuelve el Ejercicio 3 aquí



<details>
<summary>Click para ver la solución del Ejercicio 3</summary>

```python
col = "Quantity"
data = df[col].dropna()

# 1. Distribución
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.histplot(data, kde=True, bins=10, color="#16A34A", alpha=0.5, ax=axes[0])
axes[0].set_title(f"Distribución de {col}\nskew = {data.skew():.3f}")

# 5. Q-Q Plot
stats.probplot(data, dist="norm", plot=axes[1])
axes[1].set_title(f"Q-Q Plot de {col}")

# 5. Shapiro-Wilk
stat_val, p_val = shapiro(data)
axes[2].text(0.5, 0.5, f"Shapiro-Wilk\np = {p_val:.4f}\n{'Normal' if p_val > 0.05 else 'NO Normal'}",
             ha="center", va="center", fontsize=16, transform=axes[2].transAxes)
axes[2].set_xticks([]); axes[2].set_yticks([])
plt.tight_layout()
plt.show()

# 2. Centro
print(f"Media: {data.mean():.2f}")
print(f"Mediana: {data.median():.2f}")
print(f"Moda: {data.mode()[0]}")
print(f"Skew: {data.skew():.3f}")

# 3. Dispersión
print(f"\nStd: {data.std():.2f}")
q1, q3 = data.quantile(0.25), data.quantile(0.75)
print(f"IQR: {q3 - q1:.2f} (Q1={q1}, Q3={q3})")
print(f"CV: {data.std()/data.mean()*100:.1f}%")

# 4. Outliers
iqr = q3 - q1
outliers = data[(data < q1-1.5*iqr) | (data > q3+1.5*iqr)]
print(f"\nOutliers: {len(outliers)}")

# 6. Información para el gerente
print("\n--- PARA EL GERENTE ---")
print(f"La mayoría de clientes compran entre {q1:.0f} y {q3:.0f} productos por visita,")
print(f"con una cantidad típica de {data.median():.0f} productos.")
print(f"La distribución es uniforme (no hay un pico claro),")
print(f"lo que significa que hay demanda variada — desde compras")
print(f"pequeñas (1-2 items) hasta grandes (8-10 items).")
```
</details>

---
## Resumen de la Clase

| Concepto | Lo que aprendimos | Código clave |
|---------|------------------|-------------|
| **Distribución** | Cómo se reparten los valores. La forma decide qué herramientas usar | `sns.histplot(kde=True)` |
| **Sesgo (skewness)** | Mide asimetría. Si \|skew\| > 0.5, la media no es confiable | `.skew()` |
| **Media** | Centro sensible a outliers. Usar cuando la distribución es simétrica | `.mean()` |
| **Mediana** | Centro robusto. Usar cuando hay sesgo u outliers | `.median()` |
| **Moda** | Valor más frecuente. Usar con texto/categorías | `.mode()[0]` |
| **Std** | Dispersión típica, mismas unidades que los datos | `.std()` |
| **IQR** | Dispersión del 50% central. Robusta a outliers | `.quantile(.75) - .quantile(.25)` |
| **CV** | Dispersión relativa. Compara variables con unidades distintas | `std / mean * 100` |
| **Outlier (IQR)** | Datos fuera de [Q1−1.5×IQR, Q3+1.5×IQR] | Filtro booleano |
| **Z-score** | Distancia en $\sigma$ de la media. \|z\| > 3 = outlier fuerte | `(x - mean) / std` |
| **Normal** | Campana de Gauss. Regla 68-95-99.7 | Shapiro-Wilk, Q-Q plot |
| **Dato → Información** | Pregunta "¿Y entonces qué?" para convertir números en decisiones | Contexto de negocio |

### Próxima clase (lunes 30)
- Correlación: ¿hay relación entre dos variables?
- Scatterplot y coeficiente de Pearson
- Heatmap de correlación